# Day 2 — Data Engineering
## Healthcare Fraud Detection
**UGASS DSC x Zindi Capstone**

---
## 1. Setup & Load Data

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

CSV_PATH  = r'C:\Users\Ferdinard\Desktop\CODES\mlp\data\healthcare_fraud_detection.csv'
PLOTS_DIR = r'C:\Users\Ferdinard\Desktop\CODES\mlp\plots'
DATA_DIR  = r'C:\Users\Ferdinard\Desktop\CODES\mlp\data'
os.makedirs(PLOTS_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
print('Raw data shape:', df.shape)
df.head()

Raw data shape: (10000, 20)


,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0


---
## 2. Drop Useless Columns

In [11]:
df = df.drop(columns=['Claim_ID', 'Claim_Submission_Date'])
print('Shape after dropping ID/date columns:', df.shape)
print('Remaining columns:', df.columns.tolist())

Shape after dropping ID/date columns: (10000, 18)
Remaining columns: ['Provider_ID', 'Patient_Age', 'Patient_Gender', 'Diagnosis_Code', 'Procedure_Code', 'Claim_Amount', 'Approved_Amount', 'Insurance_Type', 'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly', 'Provider_Specialty', 'Patient_State', 'Claim_Status', 'Is_Fraud', 'Length_of_Stay', 'Visit_Type', 'Chronic_Condition_Flag', 'Prior_Visits_12m']


---
## 3. Handle Missing Values

In [12]:
print('Missing values BEFORE:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df['Insurance_Type']     = df['Insurance_Type'].fillna(df['Insurance_Type'].mode()[0])
df['Provider_Specialty'] = df['Provider_Specialty'].fillna(df['Provider_Specialty'].mode()[0])
df['Prior_Visits_12m']   = df['Prior_Visits_12m'].fillna(df['Prior_Visits_12m'].median())

print('\nMissing values AFTER:', df.isnull().sum().sum(), '<- should be 0')

Missing values BEFORE:
Insurance_Type        350
Provider_Specialty    350
Prior_Visits_12m      350
dtype: int64

Missing values AFTER: 0 <- should be 0


---
## 4. Feature Engineering — Create New Features

In [13]:
# Feature 1: Claim-to-Approved Ratio
df['claim_to_approved_ratio'] = df['Claim_Amount'] / (df['Approved_Amount'] + 1)
print('Claim-to-Approved Ratio by Fraud label:')
print(df.groupby('Is_Fraud')['claim_to_approved_ratio'].mean().round(3))

# Feature 2: Provider Fraud Rate
provider_fraud_rate = df.groupby('Provider_ID')['Is_Fraud'].mean()
df['provider_fraud_rate'] = df['Provider_ID'].map(provider_fraud_rate)
print('\nProvider Fraud Rate by Fraud label:')
print(df.groupby('Is_Fraud')['provider_fraud_rate'].mean().round(4))

# Feature 3: Claim Per Day
df['claim_per_day'] = df['Claim_Amount'] / (df['Length_of_Stay'] + 1)
print('\nClaim Per Day by Fraud label:')
print(df.groupby('Is_Fraud')['claim_per_day'].mean().round(2))

print('\nAll 3 engineered features added successfully!')

Claim-to-Approved Ratio by Fraud label:
Is_Fraud
0    1.146
1    1.965
Name: claim_to_approved_ratio, dtype: float64

Provider Fraud Rate by Fraud label:
Is_Fraud
0    0.0781
1    0.1361
Name: provider_fraud_rate, dtype: float64

Claim Per Day by Fraud label:
Is_Fraud
0    238.29
1    451.08
Name: claim_per_day, dtype: float64

All 3 engineered features added successfully!


---
## 5. Drop Provider_ID

In [14]:
df = df.drop(columns=['Provider_ID'])
print('Shape after dropping Provider_ID:', df.shape)

Shape after dropping Provider_ID: (10000, 20)


---
## 6. Encode Categorical Variables

In [15]:
# FIX: use include='object' only — no 'str' which causes TypeError in newer pandas
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_cols)

# One-Hot Encode all categorical columns at once
df_encoded = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True
)

# Convert all boolean columns to int (0/1) — required for sklearn
bool_cols = df_encoded.select_dtypes(include='bool').columns.tolist()
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print('\nShape after encoding:', df_encoded.shape)
print('Sample columns:', df_encoded.columns.tolist()[:10])

Categorical columns to encode: ['Patient_Gender', 'Diagnosis_Code', 'Insurance_Type', 'Provider_Specialty', 'Patient_State', 'Claim_Status', 'Visit_Type']

Shape after encoding: (10000, 42)
Sample columns: ['Patient_Age', 'Procedure_Code', 'Claim_Amount', 'Approved_Amount', 'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly', 'Is_Fraud', 'Length_of_Stay', 'Chronic_Condition_Flag', 'Prior_Visits_12m']


---
## 7. Train-Test Split

In [16]:
# Separate features (X) from target (y)
X = df_encoded.drop(columns=['Is_Fraud'])
y = df_encoded['Is_Fraud']

# 80% train, 20% test — stratify keeps fraud ratio same in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} rows ({y_train.mean()*100:.1f}% fraud)')
print(f'Test set     : {X_test.shape[0]} rows ({y_test.mean()*100:.1f}% fraud)')
print('\n✓ Fraud ratio preserved in both sets thanks to stratify=y')

Training set : 8000 rows (8.3% fraud)
Test set     : 2000 rows (8.3% fraud)

✓ Fraud ratio preserved in both sets thanks to stratify=y


---
## 8. Scale Numerical Features

In [17]:
cols_to_scale = [
    'Claim_Amount', 'Approved_Amount', 'Prior_Visits_12m',
    'Days_Between_Service_and_Claim', 'Number_of_Claims_Per_Provider_Monthly',
    'Length_of_Stay', 'Patient_Age', 'claim_to_approved_ratio',
    'provider_fraud_rate', 'claim_per_day'
]

scaler = StandardScaler()

# fit_transform on train only
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print('Scaling complete!')
print('Claim_Amount mean after scaling (should be ~0):', X_train['Claim_Amount'].mean().round(4))

Scaling complete!
Claim_Amount mean after scaling (should be ~0): -0.0


---
## 9. Save Clean Datasets

In [18]:
X_train.to_csv(os.path.join(DATA_DIR, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(DATA_DIR,  'X_test.csv'),  index=False)
y_train.to_csv(os.path.join(DATA_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(DATA_DIR,  'y_test.csv'),  index=False)

print('Saved to:', DATA_DIR)
print('Files: X_train.csv, X_test.csv, y_train.csv, y_test.csv')

Saved to: C:\Users\Ferdinard\Desktop\CODES\mlp\data
Files: X_train.csv, X_test.csv, y_train.csv, y_test.csv


---
## 10. Day 2 Summary

In [19]:
print('=== DAY 2 DATA ENGINEERING SUMMARY ===')
print(f'Final feature count : {X_train.shape[1]} columns')
print(f'Training samples    : {X_train.shape[0]}')
print(f'Test samples        : {X_test.shape[0]}')
print()
print('Steps completed:')
print('  ✓ Dropped ID and date columns')
print('  ✓ Filled 350 missing values')
print('  ✓ Engineered 3 new features')
print('  ✓ One-Hot Encoded all categorical columns')
print('  ✓ Train/test split 80/20 with stratification')
print('  ✓ Scaled numerical features (no data leakage)')
print('  ✓ Saved clean datasets to /data')
print()
print('Tomorrow (Day 3): Train Logistic Regression, Random Forest, XGBoost')

=== DAY 2 DATA ENGINEERING SUMMARY ===
Final feature count : 41 columns
Training samples    : 8000
Test samples        : 2000

Steps completed:
  ✓ Dropped ID and date columns
  ✓ Filled 350 missing values
  ✓ Engineered 3 new features
  ✓ One-Hot Encoded all categorical columns
  ✓ Train/test split 80/20 with stratification
  ✓ Scaled numerical features (no data leakage)
  ✓ Saved clean datasets to /data

Tomorrow (Day 3): Train Logistic Regression, Random Forest, XGBoost
